In [1]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import community
from collections import defaultdict
import random
import copy
from search_on_graph import bge_m3_similarity

def parse_json_file(file_path):
    """
    解析JSON文件，提取relevant APIs字段。
    
    :param file_path: str - JSON文件路径
    :return: List[List[Tuple]] - 返回工具调用序列的列表
    """
    tool_sequences = []
    
    # 读取JSON文件
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    # 遍历每个查询，提取relevant APIs
    for entry in data:
        relevant_apis = entry['relevant APIs']
        tool_sequence = relevant_apis
        tool_sequences.append(tool_sequence)
    
    print(tool_sequence)  # 输出最后一条工具调用序列
    return tool_sequences

def build_graph_from_json(tool_sequences):
    """
    构建基于工具调用顺序的静态图谱，并添加start和end节点。
    
    :param tool_sequences: List[List[Tuple]] - 工具调用顺序的列表
    :return: Graph - 构建的无向图
    """
    G = nx.Graph()
    transition_counts = defaultdict(int)  # 记录工具之间的转换次数
    
    # 遍历工具调用序列，记录转换关系
    for sequence in tool_sequences:
        for i in range(len(sequence) - 1):
            tool1 = f"{sequence[i][0]}-{sequence[i][1]}"
            tool2 = f"{sequence[i + 1][0]}-{sequence[i + 1][1]}"
            transition_counts[(tool1, tool2)] += 1
            G.add_edge(tool1, tool2, weight=transition_counts[(tool1, tool2)])
    
    # 添加start和end节点
    start_node = "start"
    end_node = "end"
    
    # 创建 G 的深拷贝
    G_copy = copy.deepcopy(G)  # 深拷贝图 G
    
    # 将start节点连接到所有工具节点
    nodes_list = list(G_copy.nodes())  # 创建节点列表
    for node in nodes_list:
        G_copy.add_edge(start_node, node, weight=1)
    
    # 将end节点连接到所有工具节点
    for node in nodes_list:
        G_copy.add_edge(node, end_node, weight=1)
    
    return G_copy

def print_cluster_info(partition):
    """
    打印聚类信息，包括聚类数量、每个聚类的节点数量及具体节点

    :param partition: Dict - 节点与其聚类的映射
    """
    cluster_dict = defaultdict(list)

    # 将节点按聚类分组
    for node, cluster in partition.items():
        cluster_dict[cluster].append(node)

    # 打印聚类数量
    print(f"总共的聚类数量: {len(cluster_dict)}")
    
    # 打印每个聚类的信息
    for cluster_id, nodes in cluster_dict.items():
        print(f"聚类 {cluster_id}: 节点数量 {len(nodes)}, 节点: {nodes}")

def count_clusters(partition):
    """
    计算聚类中的簇的数量

    :param partition: Dict - 节点与其聚类的映射
    :return: int - 簇的数量
    """
    unique_clusters = set(partition.values())  # 获取唯一簇标识
    return len(unique_clusters)  # 返回唯一簇的数量

# 获取最相似的cluster
def get_most_similar_cluster(query: str, clusters: dict) -> int:
    best_cluster_id = None
    best_similarity = -1

    for cluster_id, nodes in clusters.items():
        # 计算cluster的summary
        cluster_summary = " ".join(nodes)  # 可以根据需要调整summary的定义
        similarity = bge_m3_similarity([query], [cluster_summary])[0][0]
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_cluster_id = cluster_id

    return best_cluster_id


# 示例用法
json_file_path = '/Users/caizhuoyue/Desktop/my-langgraph/data/instruction/G1_query.json'

# 解析JSON文件，提取工具调用序列
tool_sequences = parse_json_file(json_file_path)

# 构建图谱
G = build_graph_from_json(tool_sequences)

print(G)

[['eazita.com', 'Check Account Balance']]
Graph with 8482 nodes and 32162 edges


In [2]:
def print_node_set_size(G):
    """
    打印图中所有节点的集合大小。

    :param G: nx.Graph - 构建的图对象
    """
    node_set = set(G.nodes())  # 提取所有节点放入集合
    print(f"图中节点的数量: {len(node_set)}")  # 打印集合大小

print_node_set_size(G)

图中节点的数量: 8482


In [10]:
"""
遍历 tools 文件夹中的所有子文件夹，读取每个 json 文件，提取其内容并添加 tool 和 category 字段，最后将所有结果存储为一个新的 json 文件。
"""

import os
import json

def extract_and_process_tools(root_dir: str, output_file: str) -> None:
    """
    遍历 tools 文件夹的所有子文件夹，读取每个 json 文件内容，并为其添加 `tool` 和 `category` 字段，最后存储到一个新的 json 文件中。

    Args:
        root_dir (str): 根目录路径，即 "tools" 文件夹的路径。
        output_file (str): 输出的 JSON 文件名称（包括路径）。

    Returns:
        None
    """
    # 用于存储所有 tool 信息的集合
    tool_list = []

    # 遍历根目录下的所有子文件夹
    for category in os.listdir(root_dir):
        category_path = os.path.join(root_dir, category)
        # 只处理子文件夹
        print(f"正在处理:{category_path}文件夹")
        if os.path.isdir(category_path):
            # 遍历子文件夹中的所有文件
            for file_name in os.listdir(category_path):
                print(f"正在处理{file_name}文件")
                file_path = os.path.join(category_path, file_name)

                # 只处理 .json 文件
                if file_name.endswith('.json'):
                    try:
                        # 读取并解析 JSON 文件
                        with open(file_path, 'r', encoding='utf-8') as json_file:
                            json_data = json.load(json_file)

                         # "api_list": [
                         #    {
                         #        "name": "ff",
                         #        "url": "https://b4c6577cb2msh7c15fca215f2c30p1f1717jsn998498c6865e.p.rapidapi.com/",
                         #        "description": "fff",
                         #        "method": "GET",
                         #        "required_parameters": [],
                         #        "optional_parameters": [],
                         #        "code": "import requests\n\nurl = \"https://b4c6577cb2msh7c15fca215f2c30p1f1717jsn998498c6865e.p.rapidapi.com/\"\n\nheaders = {\n            \"X-RapidAPI-Key\": \"SIGN-UP-FOR-KEY\",\n            \"X-RapidAPI-Host\": \"b4c6577cb2msh7c15fca215f2c30p1f1717jsn998498c6865e.p.rapidapi.com\"\n        }\n\nresponse = requests.get(url, headers=headers)\nprint(response.json())\n",
                         #        "convert_code": "import requests\n\nurl = \"https://b4c6577cb2msh7c15fca215f2c30p1f1717jsn998498c6865e.p.rapidapi.com/\"\n\nheaders = {\n            \"X-RapidAPI-Key\": \"SIGN-UP-FOR-KEY\",\n            \"X-RapidAPI-Host\": \"b4c6577cb2msh7c15fca215f2c30p1f1717jsn998498c6865e.p.rapidapi.com\"\n        }\n\nresponse = requests.get(url, headers=headers)\nprint(response.json())\n",
                         #        "test_endpoint": ""
                         #    }

                        api_list = json_data["api_list"]
                        

                        for api in api_list:
                            api_object = {}
                            api_object["api_name"] = api["name"]
                            api_object["api_description"] = api["description"]

                            api_object["tool_name"] = json_data["tool_name"]
                            api_object["category"] = category

                            tool_list.append(api_object)
                            print(api_object)

                    except (json.JSONDecodeError, FileNotFoundError) as e:
                        print(f"文件读取错误: {file_path} - 错误信息: {e}")

    print(f"一共有{len(tool_list)}个API")

    # 将最终的 JSON 列表写入输出文件
    with open(output_file, 'w+', encoding='utf-8') as output_json:
        json.dump(tool_list, output_json, ensure_ascii=False, indent=4)
    
    print(f"所有工具信息已成功保存至 {output_file}")

    return tool_list

# 设置根目录（"tools" 文件夹）的路径和输出文件路径
ROOT_DIR = "/Users/caizhuoyue/Desktop/my-langgraph/data/toolenv/tools"
OUTPUT_FILE = "/Users/caizhuoyue/Desktop/rapidapi_all_apis.json"

# 执行工具信息提取并存储到指定文件中
tool_list = extract_and_process_tools(ROOT_DIR, OUTPUT_FILE)

正在处理:/Users/caizhuoyue/Desktop/my-langgraph/data/toolenv/tools/Commerce文件夹
正在处理fraud_prevention_web_service文件
正在处理nj_amazon_scraper文件
正在处理sandbox_ecombr_com_04_orders文件
正在处理restapitest文件
正在处理sales_tax_rates文件
正在处理blocktrail_bitcoin_developers_platform文件
正在处理bnpl_payment文件
正在处理sandbox_ecombr_com_01_products文件
正在处理jsmamazon_data_scraper文件
正在处理product_hunt文件
正在处理green_s_amazon_scrapper文件
正在处理fme_magento_testimonials_and_reviews_extension文件
正在处理apfelpreise文件
正在处理togo420.json文件
{'api_name': 'Our Catalogue', 'api_description': 'This endpoint allows developers to view our inventory catalogue with inventory quantities, product images, product descriptions, etc.', 'tool_name': 'togo420', 'category': 'Commerce'}
正在处理amazon_search_products文件
正在处理oregon_lottery文件
正在处理india_pan_card_ocr文件
正在处理barcode_verification_and_conversion文件
正在处理bring_a_trailer_scraper文件
正在处理reconditioned_apple_devices文件
正在处理togo420文件
正在处理best_buy_stock_check文件
正在处理h30_e_commerce_data_scraper文件
正在处理amazon_india_web_scraper文件
正

In [11]:
import json
from typing import List, Dict, Any
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import os

def load_tool_list(file_path: str) -> List[Dict[str, Any]]:
    """
    从指定的JSON文件加载工具列表。

    :param file_path: str - JSON文件的路径
    :return: List[Dict[str, Any]] - 工具列表
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            tool_list = json.load(file)
        print(f"成功加载工具列表，包含 {len(tool_list)} 个工具。")
        return tool_list
    except FileNotFoundError as e:
        raise RuntimeError(f"文件未找到: {e}") from e
    except json.JSONDecodeError as e:
        raise RuntimeError(f"读取JSON文件时出错: {e}") from e

def save_embeddings_to_file(embeddings: Dict[str, List[float]], file_path: str) -> None:
    """
    将嵌入保存到指定的JSON文件中。

    :param embeddings: Dict[str, List[float]] - API名称与对应嵌入的映射
    :param file_path: str - 保存嵌入的文件路径
    """
    try:
        with open(file_path, 'w', encoding='utf-8') as file:
            json.dump(embeddings, file)
        print(f"成功保存嵌入到文件: {file_path}")
    except IOError as e:
        raise RuntimeError(f"保存文件时出错: {e}") from e

def load_embeddings_from_file(file_path: str) -> Dict[str, List[float]]:
    """
    从指定的JSON文件加载API嵌入。

    :param file_path: str - JSON文件的路径
    :return: Dict[str, List[float]] - API名称与对应嵌入的映射
    """
    if not os.path.exists(file_path):
        print(f"嵌入文件 {file_path} 不存在，返回空字典。")
        return {}

    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            embeddings = json.load(file)
        print(f"成功加载嵌入，包含 {len(embeddings)} 个API。")
        return embeddings
    except FileNotFoundError as e:
        raise RuntimeError(f"文件未找到: {e}") from e
    except json.JSONDecodeError as e:
        raise RuntimeError(f"读取JSON文件时出错: {e}") from e

def bge_m3_embedding(sentences: List[str]) -> List[List[float]]:
    """
    使用BGE M3模型计算句子列表的嵌入。

    :param sentences: List[str] - 要计算嵌入的句子列表
    :return: List[List[float]] - 嵌入向量列表
    """
    model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
    embeddings = model.encode(sentences, batch_size=24, max_length=8192)['dense_vecs']
    return embeddings

def compute_tool_embeddings(tool_list: List[Dict[str, Any]], embedding_file: str) -> Dict[str, List[float]]:
    """
    计算工具列表的嵌入，如果嵌入已存在则直接加载。

    :param tool_list: List[Dict[str, Any]] - 工具列表
    :param embedding_file: str - 保存或加载嵌入的文件路径
    :return: Dict[str, List[float]] - 工具名称与对应嵌入的映射
    """
    # 尝试从文件加载嵌入
    tool_embeddings = load_embeddings_from_file(embedding_file)
    
    if tool_embeddings:
        return tool_embeddings

    print("开始计算工具嵌入...")
    tools = [f"{tool['tool_name']} {tool['api_description']}" for tool in tool_list]
    
    # 计算嵌入
    embeddings = bge_m3_embedding(tools)
    
    tool_embeddings = {tool['tool_name']: embedding for tool, embedding in zip(tool_list, embeddings)}
    print(f"总共计算了 {len(tool_embeddings)} 个工具的嵌入。")
    
    # 保存嵌入到文件
    save_embeddings_to_file(tool_embeddings, embedding_file)
    
    return tool_embeddings

def find_most_similar_api(category: str, user_query: str, tool_embeddings: Dict[str, List[float]]) -> str:
    """
    根据类别和用户查询找到最相似的API。

    :param category: str - 用户选择的类别
    :param user_query: str - 用户输入的查询
    :param tool_embeddings: Dict[str, List[float]] - 工具与嵌入的映射
    :return: str - 最相似的API名称
    """
    print(f"正在筛选类别为 '{category}' 的工具...")
    # 筛选工具
    filtered_tools = {tool['tool_name']: emb for tool, emb in zip(tool_embeddings.keys(), tool_embeddings.values()) 
                      if any(tool_item['tool_name'] == tool and tool_item['category'] == category for tool_item in tool_list)}

    if not filtered_tools:
        print(f"没有找到类别为 '{category}' 的工具。")
        return None
    
    print(f"找到 {len(filtered_tools)} 个类别为 '{category}' 的工具，开始计算相似度...")
    
    # 计算用户查询的嵌入
    user_query_embedding = bge_m3_embedding([user_query])[0]

    # 计算相似度
    similarities = {}
    for tool_name, embedding in filtered_tools.items():
        similarity = np.dot(user_query_embedding, embedding) / (np.linalg.norm(user_query_embedding) * np.linalg.norm(embedding))
        similarities[tool_name] = similarity
    
    # 找到相似度最高的工具
    most_similar_tool = max(similarities, key=similarities.get)
    print(f"最相似的API是: {most_similar_tool}，相似度为: {similarities[most_similar_tool]:.4f}")
    return most_similar_tool

# 主程序
tool_list = load_tool_list('rapidapi_all_apis.json')  # 从文件加载工具列表
embedding_file_path = '/Users/caizhuoyue/Desktop/my-langgraph/rapidapi_all_apis.json'  # 指定保存或加载嵌入的文件路径
tool_embeddings = compute_tool_embeddings(tool_list, embedding_file_path)
selected_category = "Weather"  # 这里假设用户选择的类别为 Email
user_query = "我需要搜索北京的天气预报"  # 用户的查询

most_similar_api = find_most_similar_api(selected_category, user_query, tool_embeddings)
print(f"最终选择的API: {most_similar_api}")

成功加载工具列表，包含 16464 个工具。


RuntimeError: 读取JSON文件时出错: Expecting value: line 2 column 27 (char 28)

In [9]:
import json
from typing import List, Dict, Any
from FlagEmbedding import BGEM3FlagModel
import numpy as np

# 假设 tool_list 是从某个地方加载的
# tool_list = [
#     {
#         "api_name": "getImapSmtpAccess",
#         "api_description": "Get IMAP and SMTP access usernames and passwords",
#         "tool_name": "MailSlurp Email Testing",
#         "category": "Email"
#     }
#     # 可以添加更多工具
# ]

def bge_m3_embedding(sentences: List[str]) -> List[List[float]]:
    """
    使用BGE M3模型计算句子列表的嵌入。

    :param sentences: List[str] - 要计算嵌入的句子列表
    :return: List[List[float]] - 嵌入向量列表
    """
    model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
    embeddings = model.encode(sentences, batch_size=24, max_length=8192)['dense_vecs']
    return embeddings

def compute_tool_embeddings(tool_list: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """
    计算工具列表的嵌入。

    :param tool_list: List[Dict[str, Any]] - 工具列表
    :return: Dict[str, Dict[str, Any]] - 工具名称与对应嵌入的映射
    """
    print("开始计算工具嵌入...")
    tools = [f"{tool['tool_name']} {tool['api_description']}" for tool in tool_list]
    
    # 计算嵌入并打印进度信息
    embeddings = bge_m3_embedding(tools)
    print("工具嵌入计算完成！")
    
    # 创建包含类别和嵌入的映射
    tool_embeddings = {
        tool['tool_name']: {
            'embedding': embedding,
            'category': tool['category']
        } 
        for tool, embedding in zip(tool_list, embeddings)
    }
    print(f"总共计算了 {len(tool_embeddings)} 个工具的嵌入。")
    
    return tool_embeddings

def find_most_similar_api(category: str, user_query: str, tool_embeddings: Dict[str, Dict[str, Any]]) -> str:
    """
    根据类别和用户查询找到最相似的API。

    :param category: str - 用户选择的类别
    :param user_query: str - 用户输入的查询
    :param tool_embeddings: Dict[str, Dict[str, Any]] - 工具与嵌入的映射
    :return: str - 最相似的API名称
    """
    print(f"正在筛选类别为 '{category}' 的工具...")
    # 筛选工具
    filtered_tools = {name: emb['embedding'] for name, emb in tool_embeddings.items() if emb['category'] == category}
    
    if not filtered_tools:
        print(f"没有找到类别为 '{category}' 的工具。")
        return None
    
    print(f"找到 {len(filtered_tools)} 个类别为 '{category}' 的工具，开始计算相似度...")
    
    # 计算用户查询的嵌入
    user_query_embedding = bge_m3_embedding([user_query])[0]

    # 计算相似度
    similarities = {}
    for tool_name, embedding in filtered_tools.items():
        similarity = np.dot(user_query_embedding, embedding) / (np.linalg.norm(user_query_embedding) * np.linalg.norm(embedding))
        similarities[tool_name] = similarity
    
    # 找到相似度最高的工具
    most_similar_tool = max(similarities, key=similarities.get)
    print(f"最相似的API是: {most_similar_tool}，相似度为: {similarities[most_similar_tool]:.4f}")
    return most_similar_tool

# 主程序
print("fak")
tool_embeddings = compute_tool_embeddings(tool_list)
selected_category = "Email"  # 这里假设用户选择的类别为 Email
user_query = "我需要获取 IMAP 和 SMTP 访问的用户名和密码"  # 用户的查询

most_similar_api = find_most_similar_api(selected_category, user_query, tool_embeddings)
print(f"最终选择的API: {most_similar_api}")

fak
开始计算工具嵌入...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

工具嵌入计算完成！
总共计算了 1 个工具的嵌入。
正在筛选类别为 'Email' 的工具...
找到 1 个类别为 'Email' 的工具，开始计算相似度...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

最相似的API是: MailSlurp Email Testing，相似度为: 0.8174
最终选择的API: MailSlurp Email Testing


In [ ]:
def sample_graph(G, sample_size=20):
    """
    从图中采样指定数量的节点（包括start和end节点）。
    
    :param G: Graph - 要采样的图
    :param sample_size: int - 采样的节点数量
    :return: Graph - 采样后的子图
    """
    # 确保包含start和end节点
    sampled_nodes = {"start", "end"}
    
    # 从其他节点中随机采样
    other_nodes = list(G.nodes())
    other_nodes.remove("start")
    other_nodes.remove("end")
    
    # 如果其他节点数量不足，使用所有其他节点
    if len(other_nodes) <= (sample_size - 2):
        sampled_nodes.update(other_nodes)
    else:
        sampled_nodes.update(random.sample(other_nodes, sample_size - 2))
    
    # 创建子图
    sampled_graph = G.subgraph(sampled_nodes)
    
    return sampled_graph

sampled_G = sample_graph(G, sample_size=100)

In [ ]:
def get_top_k_similar_apis(query: str, clusters: dict, k: int) -> list:
    # 存储簇的相似度与ID
    similarities = []

    for cluster_id, nodes in clusters.items():
        # 计算cluster的summary
        cluster_summary = " ".join(nodes)  # 可以根据需要调整summary的定义
        similarity = bge_m3_similarity([query], [cluster_summary])[0][0]
        similarities.append((similarity, cluster_id, nodes))  # 存储相似度、ID和内容

    # 按照相似度排序并取前k个
    top_k = sorted(similarities, key=lambda x: x[0], reverse=True)[:k]
    
    return top_k  # 返回包含相似度、ID和内容的元组

In [ ]:

start_node = "start"
query = "what's the weather like in Beijing today?"

